# 02b - Adding 2011 as a Third Historical Data Point

**Goal:** extend the 2016↔2021 ward panel (`02_cleaning_and_merge`) with 2011 results, so that legitimate delta/growth-rate features can be built.

**Why this notebook is separate from `02_cleaning_and_merge`:** that notebook is the clean, committed record of the 2016/2021 pipeline and is left untouched. This notebook reuses its logic via `src/features.py` and `src/data_loading.py` rather than redefining anything inline - this is deliberate: a redefinition of `to_ward_level` was accidentally made during ad-hoc exploration while building this extension, silently missing its ballot-type filter, and it passed several indirect consistency checks before being caught. Importing one shared copy instead of retyping it is what prevents that from happening again.

**Why 2011 at all:** with only two election years (2016, 2021), any delta feature (e.g. turnout change) computed between them requires knowing 2021 - the very thing being predicted - which is direct label leakage. A third year lets us build a delta from **two years both prior to the prediction target** (2011→2016 predicting 2021), which mirrors the deployment-time delta (2016→2021 predicting 2026) without leaking the label.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.data_loading import load_province_files
from src.features import to_ward_level, assert_ward_matches_raw

pd.set_option("display.max_columns", None)


## Step 1 - Load the 2011 raw files

**Encoding note:** unlike the 2016/2021 files (UTF-8), the 2011 export is UTF-16 (little-endian, with a BOM - confirmed by peeking at the first bytes: `b'\xff\xfeP\x00r\x00o\x00v\x00'`, which decodes as `Prov...` under UTF-16). Older IEC exports predate the standardization used from 2016 onward.

In [2]:
df_2011 = load_province_files("../data/raw/LGE2011", encoding="utf-16")
print("2011 rows:", len(df_2011))


../data/raw/LGE2011: found 9 province files (encoding=utf-16)
2011 rows: 474386


## Step 2 - Inspect structure before assuming it matches 2016/2021

In [3]:
print("BallotType values:", df_2011["BallotType"].unique())

pd.set_option("display.max_colwidth", None)
print("Sample municipality string:", df_2011["Municipality"].iloc[0])
print("Unique municipalities:", df_2011["Municipality"].nunique())


BallotType values: <ArrowStringArray>
['PR', 'Ward', 'DC 40%']
Length: 3, dtype: str
Sample municipality string: BUF - Buffalo City Metropolitan Municipality [East London]
Unique municipalities: 234


**Note on 2011 municipality naming:** 2011 municipality strings are more verbose than 2016/2021 (e.g. `"BUF - Buffalo City Metropolitan Municipality [East London]"` vs `"BUF - Buffalo City"`). This does not affect the merge, since the join key is `Province` + `Ward`, not `Municipality` - the same reasoning already applied to the Camdeboo → Dr. Beyers Naude rename between 2016 and 2021.

## Step 3 - Collapse to ward level, using the shared `to_ward_level`

Reusing the imported function (rather than a local redefinition) is the whole point of this refactor - it guarantees the same, validated ballot-type filter and aggregation logic that produced the correct 2016/2021 turnout figures.

In [4]:
ward_2011 = to_ward_level(df_2011)
print("2011 wards:", ward_2011["Ward"].nunique())


2011 wards: 4277


## Step 4 - Regression test

Run this immediately after every call to `to_ward_level`, for every year, from now on. It recomputes one ward directly from raw rows and compares it to the function's output - the one check that would have caught the missing-filter bug in seconds instead of the multi-step investigation it actually took.

In [5]:
assert_ward_matches_raw(df_2011, ward_2011, "Ward 29200001")


Sanity check passed for Ward 29200001.


## Step 5 - Load the existing 2016↔2021 panel and check 2011 overlap

We merge 2011 onto the already-validated `ward_panel.csv` (2016↔2021), rather than rebuilding 2016 and 2021 from raw data again here - that work is already done and committed in `02_cleaning_and_merge.ipynb`.

In [6]:
ward_panel_2yr = pd.read_csv("../data/processed/ward_panel.csv")
print(ward_panel_2yr.shape)
ward_panel_2yr.head()


(4344, 11)


,Province,Ward,MunicipalityCode,Municipality_2016,Municipality_2021,RegisteredVoters_2016,VotesCast_2016,Turnout_2016,RegisteredVoters_2021,VotesCast_2021,Turnout_2021
0,Eastern Cape,Ward 29200001,BUF,BUF - Buffalo City,BUF - Buffalo City,8851,5053,0.570896,9589,3840,0.400459
1,Eastern Cape,Ward 29200002,BUF,BUF - Buffalo City,BUF - Buffalo City,7794,3636,0.466513,7655,3381,0.441672
2,Eastern Cape,Ward 29200003,BUF,BUF - Buffalo City,BUF - Buffalo City,8118,3385,0.416975,9961,3372,0.338520
3,Eastern Cape,Ward 29200004,BUF,BUF - Buffalo City,BUF - Buffalo City,9175,6186,0.674223,9327,4584,0.491476
4,Eastern Cape,Ward 29200005,BUF,BUF - Buffalo City,BUF - Buffalo City,9228,5171,0.560360,8732,3526,0.403802


In [7]:
overlap_2011 = set(ward_2011["Ward"]) & set(ward_panel_2yr["Ward"])
print(f"2011 wards: {ward_2011['Ward'].nunique()}, "
      f"2yr-panel wards: {len(ward_panel_2yr)}, "
      f"overlap: {len(overlap_2011)}")


2011 wards: 4277, 2yr-panel wards: 4344, overlap: 3834


**Expect a bigger drop than the 2016↔2021 overlap (~99%/97%).** South Africa underwent a large municipal amalgamation process ahead of the 2016 LGE, so more wards are expected to fail to match between 2011 and 2016 than between 2016 and 2021.

In [8]:
# Spot-check non-matching wards look like genuine demarcation
# differences, not parsing artifacts.
missing_from_2yr = set(ward_2011["Ward"]) - set(ward_panel_2yr["Ward"])
print(ward_2011[ward_2011["Ward"].isin(missing_from_2yr)][["Province", "Municipality", "Ward"]].head(10))


         Province                     Municipality           Ward
63   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003001
64   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003002
65   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003003
66   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003004
99   Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007001
100  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007002
101  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007003
102  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007004
208  Eastern Cape  EC124 - Amahlathi [Stutterheim]  Ward 21204016
209  Eastern Cape  EC124 - Amahlathi [Stutterheim]  Ward 21204017


## Step 6 - Merge into a three-year panel

In [9]:
ward_2011_renamed = ward_2011.rename(columns={
    "Municipality": "Municipality_2011",
    "RegisteredVoters": "RegisteredVoters_2011",
    "SpoiltVotes": "SpoiltVotes_2011",
    "TotalValidVotes": "TotalValidVotes_2011",
    "VotesCast": "VotesCast_2011",
    "Turnout": "Turnout_2011",
})

ward_panel_3yr = ward_2011_renamed.merge(
    ward_panel_2yr,
    on=["Province", "Ward"],
)
print(ward_panel_3yr.shape)
ward_panel_3yr.columns.tolist()

(3834, 17)


['Province',
 'Municipality_2011',
 'Ward',
 'RegisteredVoters_2011',
 'SpoiltVotes_2011',
 'TotalValidVotes_2011',
 'VotesCast_2011',
 'Turnout_2011',
 'MunicipalityCode',
 'Municipality_2016',
 'Municipality_2021',
 'RegisteredVoters_2016',
 'VotesCast_2016',
 'Turnout_2016',
 'RegisteredVoters_2021',
 'VotesCast_2021',
 'Turnout_2021']

## Step 7 - Outlier check on 2011 turnout

Same [0.1, 1.0] plausibility bounds used for 2016/2021. **Investigate every flagged ward by tracing back to its raw rows** - the same approach used for the Limpopo (LIM345) and Matlosana cases in `02_cleaning_and_merge` - rather than dropping rows blind. Do not assume a large flagged count means real anomalies without checking; it may indicate a pipeline issue, the same way the pre-fix version of this notebook flagged 3,720 of 3,834 wards before the missing filter was found.

In [10]:
outliers_2011 = ward_panel_3yr[
    (ward_panel_3yr["Turnout_2011"] > 1.0) | (ward_panel_3yr["Turnout_2011"] < 0.05)
]
print(len(outliers_2011))
outliers_2011[["Province", "Municipality_2011", "Ward", "RegisteredVoters_2011", "VotesCast_2011", "Turnout_2011"]]

0


,Province,Municipality_2011,Ward,RegisteredVoters_2011,VotesCast_2011,Turnout_2011


In [11]:
# TODO once outliers above are investigated and understood:
# add each excluded ward here with a short, specific reason
# (mirroring the LIM345 / Matlosana documentation style from
# 02_cleaning_and_merge), then rebuild ward_panel_3yr_clean.

exclude_wards_2011 = [
    # "Ward XXXXXXXX",  # reason
]

ward_panel_3yr_clean = ward_panel_3yr[
    ~ward_panel_3yr["Ward"].isin(exclude_wards_2011)
].copy()
print(ward_panel_3yr_clean.shape)
ward_panel_3yr_clean[["Turnout_2011", "Turnout_2016", "Turnout_2021"]].describe()


(3834, 17)


,Turnout_2011,Turnout_2016,Turnout_2021
count,3834.000000,3834.000000,3834.000000
mean,0.575649,0.571184,0.466297
std,0.080639,0.077937,0.086649
min,0.168292,0.157243,0.136051
25%,0.524463,0.521265,0.408742
50%,0.576540,0.568401,0.460772
75%,0.628816,0.618695,0.520352
max,0.920731,0.857858,0.855718


## Step 8 - Build the legitimate delta features

**Correct delta definition, now that a third year exists:** `Turnout_2016 - Turnout_2011` (and the equivalent registration growth rate) predicts `Turnout_2021`. This is symmetric with deployment: `Turnout_2021 - Turnout_2016` will predict `Turnout_2026`. **Do not use `Turnout_2021 - Turnout_2016` as a training feature** - that still requires knowing 2021, the training label itself.

In [12]:
ward_panel_3yr_clean["TurnoutDelta_2011_2016"] = (
    ward_panel_3yr_clean["Turnout_2016"] - ward_panel_3yr_clean["Turnout_2011"]
)
ward_panel_3yr_clean["RegistrationGrowth_2011_2016"] = (
    ward_panel_3yr_clean["RegisteredVoters_2016"] / ward_panel_3yr_clean["RegisteredVoters_2011"] - 1
)

## Step 9 - Add municipality code and save

**Saved as a separate file** (`ward_panel_3yr.csv`), not overwriting `ward_panel.csv` - Person B may already be building features on the two-year panel, and this avoids silently swapping their input out from under them. Confirm with the team which panel becomes the modelling input going forward.

In [13]:
ward_panel_3yr_clean["MunicipalityCode"] = (
    ward_panel_3yr_clean["Municipality_2021"].str.split(" - ").str[0]
)

output_cols = [
    "Province", "Ward", "MunicipalityCode",
    "RegisteredVoters_2011", "VotesCast_2011", "Turnout_2011",
    "RegisteredVoters_2016", "VotesCast_2016", "Turnout_2016",
    "RegisteredVoters_2021", "VotesCast_2021", "Turnout_2021",
    "TurnoutDelta_2011_2016", "RegistrationGrowth_2011_2016",
]
ward_panel_3yr_final = ward_panel_3yr_clean[output_cols]
ward_panel_3yr_final.to_csv("../data/processed/ward_panel_3yr.csv", index=False)
print(f"Saved {len(ward_panel_3yr_final)} wards to data/processed/ward_panel_3yr.csv")


Saved 3834 wards to data/processed/ward_panel_3yr.csv


## Handoff notes

**File:** `data/processed/ward_panel_3yr.csv` - one row per ward, matched across all three of 2011, 2016, and 2021.

**New columns beyond the 2-year panel:** `RegisteredVoters_2011`, `VotesCast_2011`, `Turnout_2011`, `TurnoutDelta_2011_2016`, `RegistrationGrowth_2011_2016`.

**Correct feature usage:** `TurnoutDelta_2011_2016` and `RegistrationGrowth_2011_2016` are safe model features for predicting `Turnout_2021`. The equivalent `_2016_2021` deltas are **not** safe as features (they require the label) - use them for EDA/storytelling only.

**Additional limitations to carry into the write-up:**
- 2011 source file required UTF-16 decoding, unlike 2016/2021 (UTF-8).
- Ward matching between 2011 and 2016 was lower than between 2016 and 2021: 89.7% of 2011 wards (3,834/4,277) and 88.3% of the 2016↔2021 panel (3,834/4,344) matched, consistent with the municipal amalgamation process that preceded the 2016 LGE.
- No turnout outliers were found in the 2011 data (checked against the same [0.1, 1.0] plausibility bounds used for 2016/2021) — no exclusions were required.
- A ballot-type filter was briefly, accidentally omitted during development of this extension and caught via the regression test in Step 4 before it reached the committed panel; `to_ward_level` was moved into `src/features.py` as a single shared, tested source of truth to prevent recurrence.